# Bài 10: Tích chập (Convolution)

Tích chập (convolution) là phép toán cốt lõi đằng sau làm mờ, làm sắc nét và phát hiện biên &mdash; và cũng là phép toán nằm ở trung tâm các lớp tích chập của mạng nơ-ron tích chập (Bài 34). Bài này xây dựng tích chập từ đầu, làm rõ một điểm hay gây nhầm lẫn (tích chập và tương quan), và giới thiệu một vài bộ lọc kinh điển.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

## Tích chập 1D

Trước khi xử lý ảnh 2D, hãy bắt đầu với bài toán đơn giản hơn là tích chập hai mảng 1D. Với tín hiệu $I$ và kernel $K$, tích chập rời rạc là

$$(I * K)(x) = \sum_{i} K(i)\, I(x - i)$$

Cụ thể: **lật** kernel từ đầu sang cuối, **trượt** nó dọc theo tín hiệu mỗi lần một vị trí, và tại mỗi vị trí lấy tích vô hướng giữa kernel đã lật và đoạn tín hiệu chồng lên nó. Giờ hãy thử với một tín hiệu tăng dần rồi giảm xuống, và một kernel 3 phần tử đơn giản $[-1, 0, 1]$ &mdash; một phép đạo hàm rời rạc tính độ dốc.

In [ ]:
def convolve1d(signal, kernel):
    ksize = len(kernel)
    pad = ksize // 2
    padded = np.pad(signal, pad, mode='constant')
    flipped = kernel[::-1]
    out = np.zeros_like(signal, dtype=np.float64)
    for n in range(len(signal)):
        window = padded[n:n + ksize]
        out[n] = np.dot(window, flipped)
    return out

signal = np.array([1, 2, 3, 4, 5, 4, 3, 2, 1], dtype=np.float64)
kernel = 0.5 * np.array([1, 0, -1], dtype=np.float64) # pre-flipped; scaling factor ensures that result is slope

mine = convolve1d(signal, kernel)
reference = np.convolve(signal, kernel, mode='same')
print('signal:   ', signal)
print('mine:     ', mine)
print('np.convolve:', reference)
print('match:', np.allclose(mine, reference))

fig, axes = plt.subplots(1, 2, figsize=(8, 3))
axes[0].plot(signal, marker='o')
axes[0].set_title('Signal')
axes[1].plot(mine, marker='o', color='tab:orange')
axes[1].axhline(0, color='gray', linewidth=0.8)
axes[1].set_title('Signal * [1, 0, -1]')
plt.tight_layout()
plt.show()

Lưu ý rằng đầu ra dương khi tín hiệu đang tăng, bằng 0 ở đỉnh, và âm khi tín hiệu đang giảm &mdash; đúng như những gì một kernel kiểu đạo hàm cần làm.

Dưới đây là cùng phép tính đó dưới dạng ảnh động từng bước: kernel được lật trước, rồi kernel đã lật trượt qua tín hiệu đã đệm số 0 (zero-padded) mỗi lần một vị trí, lấy tích vô hướng ở mỗi bước để tạo ra một giá trị đầu ra.

<img src="../animations/convolution_demo.gif" alt="Ảnh động minh họa tích chập 1D: lật kernel và trượt nó qua tín hiệu, tính tích vô hướng ở mỗi vị trí" style="display: block; margin: 0 auto; max-width: 850px;">

## Tích chập 2D

Phiên bản 2D của tích chập (thứ thực sự được dùng trên ảnh) có cùng ý tưởng lật-và-trượt như trường hợp 1D ở trên, chỉ là trên hai trục:

$$(I * K)(x, y) = \sum_{i}\sum_{j} K(i, j)\, I(x - i,\, y - j)$$

với ảnh $I$ và kernel $K$. Cũng như phiên bản 1D, chi tiết then chốt là các dấu *trừ*: kernel được lật (xoay 180°) trước khi trượt qua ảnh. Điều này quan trọng bất cứ khi nào kernel không đối xứng.

**Tương quan (correlation)** có cùng ý tưởng nhưng không lật:

$$(I \star K)(x, y) = \sum_{i}\sum_{j} K(i, j)\, I(x + i,\, y + j)$$

Trên thực tế, hầu hết các thư viện xử lý ảnh &mdash; kể cả `cv2.filter2D` của OpenCV &mdash; thực ra cài đặt *tương quan*, không phải tích chập, dù người ta vẫn thường gọi chung là "tích chập với một kernel". Với các kernel đối xứng (lọc hộp, Gauss) hai phép này giống hệt nhau, nên sự phân biệt hiếm khi quan trọng trong thực tế. Nhưng vẫn nên biết rằng có sự khác biệt đó.

## Tích chập từ đầu

Để cài đặt tích chập 2D: lật kernel theo cả chiều ngang và chiều dọc, đệm ảnh để kernel không bao giờ chạy ra ngoài biên, rồi tại mỗi vị trí nhân từng phần tử của kernel đã lật với các điểm ảnh bên dưới nó và cộng lại.

Minh họa trong ảnh động này: lật kernel 180°, rồi trượt nó qua một ảnh nhỏ đã đệm số 0, lấy tích vô hướng với cửa sổ 3x3 chồng lên ở mỗi vị trí. Đây cũng là kernel bất đối xứng được dùng lại bên dưới trong phần so sánh tích chập với tương quan, nơi việc lật sẽ tạo ra khác biệt thấy rõ.

<img src="../animations/convolution2d_demo.gif" alt="Ảnh động minh họa tích chập 2D: lật kernel 180 độ và trượt nó qua ảnh, tính tích vô hướng ở mỗi vị trí" style="display: block; margin: 0 auto; max-width: 900px;">

## Tăng tốc tích chập: lặp qua kernel, không phải qua ảnh

Đoạn code dưới đây cài đặt trực tiếp tích chập 2D &mdash; ngoại trừ việc để tăng tốc, nó lặp qua (ít) phần tử của kernel thay vì qua (nhiều) điểm ảnh của ảnh, dịch chuyển và cộng dồn toàn bộ ảnh đã đệm cùng lúc cho mỗi trọng số kernel.

In [ ]:
def convolve2d(image, kernel, border=cv2.BORDER_REFLECT101):
    """True convolution (kernel is flipped), single-channel, float output."""
    kh, kw = kernel.shape
    pad_h, pad_w = kh // 2, kw // 2
    flipped = kernel[::-1, ::-1]
    padded = cv2.copyMakeBorder(image, pad_h, pad_h, pad_w, pad_w, border)

    out = np.zeros(image.shape, dtype=np.float64)
    for i in range(kh):  # loop over kernel values for speed (kernel is smaller than image)
        for j in range(kw):
            out += flipped[i, j] * padded[i:i + image.shape[0], j:j + image.shape[1]].astype(np.float64)
    return out

Dưới đây là mẹo đó dưới dạng ảnh động, trên cùng ảnh và kernel như trên: thay vì làm sáng một ô đầu ra ở mỗi bước, giờ mỗi bước làm sáng một trọng số kernel và cộng một bản sao đã dịch chuyển, đã nhân hệ số của *toàn bộ* ảnh vào một bộ tích lũy &mdash; nên chỉ sau 9 bước (mỗi bước một trọng số kernel) thay vì 25 bước (mỗi bước một điểm ảnh), bộ tích lũy đạt được cùng kết quả.

<img src="../animations/convolution2d_fast_demo.gif" alt="Ảnh động minh họa mẹo tích chập hiệu quả: với mỗi trọng số kernel, cộng một bản sao đã dịch chuyển và nhân hệ số của toàn bộ ảnh vào bộ tích lũy" style="display: block; margin: 0 auto; max-width: 950px;">

### Đối chiếu với OpenCV

Vì `cv2.filter2D` tính tương quan, ta đưa vào kernel *đã được lật sẵn* để có kết quả tích chập thực sự mà so sánh.

In [ ]:
test_img = np.zeros((20, 20), dtype=np.uint8)
test_img[3:17, 3:6] = 200   # a small "corner" shape (no symmetry), so flipping is visible
test_img[3:6, 3:14] = 200

asymmetric_kernel = np.array([[1, 2, -1],
                               [0, 1,  3],
                               [-2, 1, 0]], dtype=np.float64)

mine = convolve2d(test_img, asymmetric_kernel)
reference = cv2.filter2D(test_img, cv2.CV_64F, asymmetric_kernel[::-1, ::-1],
                          borderType=cv2.BORDER_REFLECT101)

print('max abs difference:', np.abs(mine - reference).max())

### Tích chập và tương quan, nhìn thấy được

Với kernel đối xứng, tích chập và tương quan cho kết quả *giống hệt nhau*. Với kernel bất đối xứng, kết quả sẽ khác &mdash; nhưng trên thực tế hầu hết các kernel bất đối xứng chỉ gây ra một phép đảo dấu đơn giản.

In [ ]:
correlated = cv2.filter2D(test_img, cv2.CV_64F, asymmetric_kernel, borderType=cv2.BORDER_REFLECT101)

fig, axes = plt.subplots(1, 3, figsize=(9, 3.5))
for ax, im, title in zip(axes, [test_img, mine, correlated],
                          ['Original', 'Convolution\n(kernel flipped)', 'Correlation\n(kernel not flipped)']):
    ax.imshow(im, cmap='gray')
    ax.set_title(title, fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

print('convolution == correlation with kernel rotated 180?',
      np.allclose(mine, cv2.filter2D(test_img, cv2.CV_64F, asymmetric_kernel[::-1, ::-1],
                                      borderType=cv2.BORDER_REFLECT101)))

## Xử lý biên

Ở gần các cạnh, kernel thò ra ngoài ảnh. `convolve2d` dùng `cv2.copyMakeBorder` để đệm trước; chế độ đệm làm thay đổi kết quả ở biên. Các lựa chọn phổ biến: đệm hằng số (số 0), sao chép (replicate — kéo dài điểm ảnh ở cạnh), và phản xạ (reflect — lấy đối xứng gương qua cạnh). Với phản xạ có hai lựa chọn: `REFLECT101` phổ biến hơn vì tránh lặp lại chính điểm ảnh ở cạnh. Trong ví dụ này, mảng $4 \times 4$ bên trong của mỗi đầu ra giống hệt mảng gốc &mdash; chỉ có vòng ngoài là khác.

In [ ]:
small = np.arange(1, 17, dtype=np.float64).reshape(4, 4)

for name, mode in [('CONSTANT (zero)', cv2.BORDER_CONSTANT),
                    ('REPLICATE', cv2.BORDER_REPLICATE),
                    ('REFLECT101', cv2.BORDER_REFLECT101)]:
    padded = cv2.copyMakeBorder(small, 1, 1, 1, 1, mode)
    print(f'{name}:\n{padded}\n')

Dưới đây là ba chế độ đó đặt cạnh nhau, với mũi tên từ mỗi ô đệm trỏ về điểm ảnh bên trong mà giá trị của nó được lấy từ đó (CONSTANT không có mũi tên, vì mọi ô đệm đều đơn giản bằng 0):

<img src="../figures/boundary_modes.png" alt="So sánh cạnh nhau các chế độ đệm biên CONSTANT, REPLICATE và REFLECT101, với mũi tên từ mỗi ô đệm tới điểm ảnh nguồn của nó" style="display: block; margin: 0 auto; max-width: 1000px;">

## Các bộ lọc kinh điển dưới dạng kernel

Khi đã tính được tích chập, hầu hết các bộ lọc chỉ là "chọn một kernel":

- **Lọc hộp (box blur)**: mọi điểm lân cận có trọng số bằng nhau &mdash; làm trung bình hóa nhiễu nhưng làm mờ các cạnh.
- **Làm mờ (khả tách)**: các điểm lân cận được đánh trọng số nặng hơn về phía tâm (dạng hình lều) &mdash; giảm dần mượt hơn lọc hộp, và như ta sẽ thấy bên dưới, có thể dựng nó từ hai kernel 1D.
- **Làm sắc nét (sharpen)**: tăng cường điểm ảnh trung tâm so với các điểm lân cận.
- **Sobel (phát hiện biên)**: một kernel bất đối xứng phản hồi mạnh với thay đổi cường độ theo một hướng &mdash; đây là trường hợp mà việc lật trong tích chập-so-với-tương quan thực sự có ý nghĩa, vì kernel bất đối xứng (dẫn đến đảo dấu).

In [ ]:
photo_like = np.zeros((100, 100), dtype=np.uint8)
cv2.rectangle(photo_like, (20, 20), (80, 80), 200, -1)
cv2.circle(photo_like, (50, 50), 20, 60, -1)
rng = np.random.default_rng(0)
photo_like = photo_like.astype(np.float64) + rng.normal(0, 12, photo_like.shape)
photo_like = np.clip(photo_like, 0, 255)

box = np.ones((5, 5)) / 25

k1d_int = np.array([1, 4, 6, 4, 1])  # binomial approximation to a Gaussian (Pascal's triangle, row 4)
k1d = k1d_int / k1d_int.sum()
gaussian = np.outer(k1d, k1d)

sharpen = np.array([[0, -1, 0],
                     [-1, 5, -1],
                     [0, -1, 0]], dtype=np.float64)

sobel_x = np.array([[-1, 0, 1],
                     [-2, 0, 2],
                     [-1, 0, 1]], dtype=np.float64)

results = {
    'Original (noisy)': photo_like,
    'Box blur': convolve2d(photo_like, box),
    'Blur (separable)': convolve2d(photo_like, gaussian),
    'Sharpen': convolve2d(photo_like, sharpen),
    'Sobel x (edges)': convolve2d(photo_like, sobel_x),
}

fig, axes = plt.subplots(1, 5, figsize=(15, 3.5))
for ax, (title, im) in zip(axes, results.items()):
    ax.imshow(im, cmap='gray')
    ax.set_title(title, fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

## Kernel khả tách: một mẹo tăng tốc

Kernel làm mờ ở trên là **khả tách (separable)**: nó có thể viết dưới dạng tích ngoài (outer product) của hai kernel 1D, $K = k_x \, k_y^\top$. Điều đó có nghĩa là tích chập với kernel đầy đủ $n \times n$ tương đương với tích chập với một kernel $1 \times n$, rồi một kernel $n \times 1$ &mdash; biến khối lượng công việc $O(n^2)$ trên mỗi điểm ảnh thành $O(2n)$.

In [ ]:
print('1D kernel (k1d_int), scale by 1/16:')
print(k1d_int)
print()
print('2D kernel (outer product of k1d_int), scale by 1/256:')
print(np.outer(k1d_int, k1d_int))
print()

outer_product = np.outer(k1d, k1d)
print('2D kernel == outer product of 1D kernels?', np.allclose(gaussian, outer_product))

separable_result = convolve2d(convolve2d(photo_like, k1d.reshape(1, -1)), k1d.reshape(-1, 1))
full_result = convolve2d(photo_like, gaussian)

print('separable == full 2D convolution?', np.allclose(separable_result, full_result))

Ảnh động này cho thấy kernel làm mờ 5x5 được dựng như tích ngoài của hai kernel 1D (hoặc, tương đương, coi `kx` là một ảnh hàng 1x5 và `ky` là một ảnh cột 5x1 rồi lấy tích chập 2D của chúng).

<img src="../animations/separable_kernel_demo.gif" alt="Ảnh động minh họa việc dựng kernel 2D như tích ngoài của hai kernel 1D, từng hàng một" style="display: block; margin: 0 auto; max-width: 700px;">

Ảnh động này minh họa tích chập khả tách: kernel đó được áp dụng lên ảnh ở trên dưới dạng một lượt theo hàng rồi một lượt theo cột. Tích chập khả tách đạt được cùng kết quả như tích chập trực tiếp với kernel 2D đầy đủ.

<img src="../animations/separable_pass_demo.gif" alt="Ảnh động minh họa tích chập khả tách áp dụng thành một lượt theo hàng rồi một lượt theo cột, so sánh với tích chập 2D trực tiếp" style="display: block; margin: 0 auto; max-width: 950px;">

### Bài tập

1. Xác minh rằng kernel hộp cũng khả tách, bằng cách viết nó dưới dạng tích ngoài của hai kernel 1D đều.
2. `sobel_x` ở trên không đối xứng. Tính cả tích chập lẫn tương quan của `photo_like` với nó, và mô tả hai đầu ra khác nhau thế nào (gợi ý: hãy nghĩ xem mỗi phép phản hồi với chiều thay đổi cường độ nào).
3. Đo thời gian `convolve2d` so với `cv2.filter2D` trên ảnh 300x300 với kernel Gauss 15x15 bằng `%timeit`. Sau đó đo thời gian phiên bản khả tách hai lượt so với phiên bản 2D đầy đủ. Mỗi cách tăng tốc nhanh hơn bao nhiêu?